# 10_00 Advanced: your own domain (overview)

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: the same pattern, on a company's own text
>
> Project files this connects to:
>
> - `src/homer_ai/domain.py`
> - `src/homer_ai/domain_server.py`
> - `data/domain/`
> - `scripts/make_domain_data.py`
> - `scripts/benchmark_domain.py`
>
> **This notebook:** Homer was practice. Why would a company want its own LLM, what does that usually mean, and what does the course's pattern look like on a firm's own documents, emails and tickets?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a company usually means when it says it wants its own LLM
- The four rungs again, this time with the reasons a firm climbs each one
- The three worked examples of this chapter, and the invented data they run on
- What results to expect, and how far to trust numbers measured on made-up data

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **hosted model** | a model that runs on its maker's computers: you send text over the internet and get text back | Claude, through an app or an API |
| **open model** | a model whose weights you can download and run on a machine of your own | GPT-2 small, the 548 MB file of chapter 8 |
| **synthetic data** | data that somebody made up, by hand or with a program, to stand in for real data | 400 emails built from lists of sentences |
| **escalation** | handing a conversation from the program to a person | a complaint goes to the complaints team |
| **human in the loop** | a person checks or decides before the result counts | the model flags an email; a person decides what it is |

## 3. The idea

### ❓ Why would a company need its OWN LLM?

Mostly it doesn't, not in the sense of this course's first seven chapters. Here's the picture I use.

Your firm needs help with its paperwork. There are four ways to get it.

1. You phone a brilliant outside expert. They know a lot about the world and nothing about your firm, and every question has to be read out to them over the phone.
2. You give that same expert your binder of documents and a few phone extensions they may call. Now they look things up before they answer, and they can tell you the page.
3. You hire a junior clerk and train them for one job, say sorting the post. They sit in your building, they cost little per letter, and they do the job the same way every day.
4. You raise a child and school them yourself until they can do the job.

| In the picture | In the course | Where |
|---|---|---|
| the outside expert on the phone | **prompting** a big hosted model | what you do every day with Claude |
| the expert with your binder and your extensions | **RAG and tools over MCP** | chapters 6 and 9, and [10_01](10_01_support_with_citations.ipynb), [10_03](10_03_service_desk_triage.ipynb) |
| the clerk you trained for one job | **fine-tuning** a small open model | chapter 8, and [10_02](10_02_spotting_complaints.ipynb) |
| the child you raised | **training from scratch** | chapters 1 to 7 |

So when a company says *we want our own LLM*, it almost never means number 4. It means 2 or 3: a model that answers from **our** documents, or a small one that runs on **our** machines.

The picture has a limit, and in this chapter it matters. An expert and a clerk understand what they read, and they can be held responsible for what they say. A model does neither. Whoever puts it in front of a customer stays responsible for every word, so the examples here spend as much time on when the model must stay quiet as on what it says.

This is the ladder of [08_00](../08_advanced_fine_tuning/08_00_chapter_overview.ipynb). The next cell draws it with the reasons beside each rung.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    EDGE,
    GOOD,
    INK,
    MUTED,
    arrow,
    diagram_axes,
    stack_of_floors,
    studio_figure,
    tag,
)

fig = studio_figure(
    "Your domain", "Four rungs: climb only as far as you need", (10, 6.8)
)
ax = diagram_axes(fig)

# (title, what it is, colour), from the bottom rung up
rungs = [
    ("1. Prompt a big model", "ask well; nothing of yours is inside it", EDGE),
    (
        "2. RAG + tools over MCP",
        "your documents and systems, looked up for each answer",
        GOOD,
    ),
    (
        "3. Fine-tune a small open model",
        "your examples change the weights; it runs on your machines",
        ACCENT,
    ),
    ("4. Train from scratch", "a model of your own, from nothing", EDGE),
]
# why a firm takes each step
reasons = [
    ("start here", MUTED),
    ("exact answers, each with its source\n10_01 and 10_03", GOOD),
    (
        "data that can't leave, cost at volume,\na frozen model for audits: 10_02",
        ACCENT,
    ),
    ("rare: it needs more text and money\nthan most firms have", MUTED),
]
middles = stack_of_floors(ax, 3.1, 0.35, 5.4, rungs, floor_h=1.05, gap=0.22)
tag(ax, 8.05, 5.6, "you climb for…", INK, 10.5, weight="bold")
for y, (text, color) in zip(middles, reasons, strict=True):
    arrow(ax, (5.95, y), (6.45, y), color=color)
    tag(ax, 8.05, y, text, color, 9)
arrow(ax, (0.2, 0.5), (0.2, 5.2), color=MUTED, width=2)
ax.text(
    0.02,
    2.85,
    "more cost, more work",
    color=MUTED,
    fontsize=9,
    rotation=90,
    ha="center",
    va="center",
)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Four rungs, cheapest at the bottom. On the right is what you get for climbing. Rung 2 buys exact answers with a source. Rung 3 buys a model that stays inside the building, costs little per text, and doesn't change unless you change it. Rungs 2 and 3 are the coloured ones: that's where this chapter works.

</details>

### ❓ What makes a firm climb, in practice?

Four reasons come up again and again. None of them is about being clever.

**Exact answers with a source (rung 2).** A savings rate, a notice period, a deadline: one wrong digit is a wrong answer to a customer. A big model has never read your product sheet, so from memory it can only guess or decline. With the sheet in front of it, it can quote the line and say where it is. That was chapter 9, and [10_01](10_01_support_with_citations.ipynb) does it for a savings firm.

**Data that can't leave (rung 3).** Customer emails hold names, health and money worries. Some firms may not send that to anybody else's computers, and others need a long approval before they can. A small open model runs on a machine in the building. You did that on a laptop in chapter 8.

**Cost at volume (rung 3).** A hosted model is paid per text. For a few hundred questions a day that's nothing. For every email, chat line and call note of a large firm it adds up, while a small model on your own machine costs the same whether it reads a hundred emails or a million. The next cell does the sum.

**A frozen model for audits (rung 3).** A hosted model is updated and, one day, retired by its maker. If somebody asks *why was this email not flagged in March?*, you need the model that ran in March. A weights file you own has a fingerprint (the SHA-256 of [08_01](../08_advanced_fine_tuning/08_01_loading_gpt2.ipynb)) and stays the same until you replace it.

Every rung also costs something. On rung 2 somebody has to keep the documents and the tools right. On rung 3 you need labelled examples, and a person who retrains the model when the world changes.

👉 **Predict first:** the cell below compares a hosted model and a machine of your own, with made-up prices. At how many emails a day do the two cost the same?

In [ ]:
# Made-up prices: put in your own. None of these is anybody's real price.
hosted_per_1000_emails = 2.00  # what a hosted model would charge to read 1,000 emails
own_machine_per_month = 400.00  # one server that runs a small model of your own
emails_per_second = 70  # measured in 10_02 on a laptop CPU; your machine will differ

print(
    f"{'emails a day':>12} {'hosted, a month':>16} {'own machine':>12} "
    f"{'machine busy':>14}"
)
for per_day in (200, 2_000, 20_000, 200_000):
    hosted = per_day * 30 / 1000 * hosted_per_1000_emails
    busy_minutes = per_day / emails_per_second / 60
    print(
        f"{per_day:>12,} {hosted:>16,.0f} {own_machine_per_month:>12,.0f} "
        f"{busy_minutes:>9.1f} min/day"
    )
same = own_machine_per_month / 30 / hosted_per_1000_emails * 1000
print(f"\nwith these prices, the two cost the same at {same:,.0f} emails a day")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The hosted column grows with the volume: 12 a month at 200 emails a day, 12,000 at 200,000. The machine column doesn't move. With these made-up prices they cross at about 6,700 emails a day.

Look at the last column too. A small model on one CPU gets through 200,000 short emails in under an hour. Speed is not the problem on rung 3.

Two honest notes. The prices are invented, so the crossing point means nothing until you put your own in. And the sum leaves out the dear part of rung 3, which is people: somebody labels the examples, somebody retrains, somebody watches the results. At low volume the hosted model wins easily, and that's why you start at the bottom of the ladder.

</details>

### ❓ Is training from scratch ever the answer?

Rarely. It needs far more text than most firms own, a lot of computing, and a team that does little else. And there are good open models to start from, so there's seldom a reason to pay for the first 99% of the learning again.

It can make sense when all three of these are true: your text is of a kind no existing model has seen much of, you have a great deal of it, and you have the means. That's a short list of organisations.

So why did this course spend seven chapters on rung 4? Because you can't judge a tool you've never seen the inside of. Every decision in this chapter leans on something from those chapters:

| A decision in this chapter | Where you learned to make it |
|---|---|
| cut an email at 64 tokens, and know what gets lost | tokens and the context window, chapter 2 |
| test on wordings the model never trained on | memorising against understanding, [05_03](../05_training_on_the_odyssey/05_03_memorising_vs_understanding.ipynb) |
| change one thing, keep a baseline, compare | fair experiments, [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb) |
| say what a tool can't do in its description | [06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb) |
| read a loss that falls on the training set only | [05_05](../05_training_on_the_odyssey/05_05_validation.ipynb) |

We built the engine once so that we can now drive with some idea of what's under the bonnet.

### ❓ What does the course's pattern look like at work?

You've done the same four moves twice already, with Homer: take **your data**, build **a tool** on it, offer the tool to an AI app over **MCP**, and **measure** with and without. This chapter does it three more times, on a company's kind of data.

| | Support ([10_01](10_01_support_with_citations.ipynb)) | Compliance ([10_02](10_02_spotting_complaints.ipynb)) | Service desk ([10_03](10_03_service_desk_triage.ipynb)) |
|---|---|---|---|
| the job | answer a customer's question about a savings account, or hand over to a person | flag emails that are complaints, or from someone who may need extra care | send a ticket to the right team and suggest a how-to article |
| rung | 2 | 3 | 2, and one try at 3 |
| your data | 4 product sheets, a FAQ, a rates table | 400 labelled emails | 200 tickets, 10 articles |
| the tool | `search_terms`, `get_rate`, `escalate` | GPT-2, fine-tuned to flag | a router and an article finder, both BM25 |
| served over MCP | yes: `domain_server.py` | no: a classifier sits in a mail pipeline, not in a chat | no |
| compared with | Claude with no tools | a list of keywords | the most common category |
| the score | right answer, right source, invented facts, handovers | **recall** on complaint and on vulnerable, then macro-F1 | routing accuracy, recall@3 for the article |

[10_04](10_04_pitfalls_and_next_steps.ipynb) then lists what would make these three demos look naive to somebody who has to run such a system for real.

### ❓ Is any of this data real?

No, and that's deliberate. Everything here is invented: the firm, its products, its customers and their words ([data/domain/README.md](../../data/domain/README.md)).

Real customer emails don't belong in a public course, and *anonymised* ones don't either: names are easy to strip, and a story about an illness or a late husband still points at a person. So I wrote the documents by hand, and a small script builds the emails and tickets from lists of sentences.

That has a price, and I'd rather you know it before the first number. Made-up text is tidier than real text. The scores in this chapter show how to measure. They don't tell you how well any of this would work on a real firm's mail.

The next cell lists what's in the folder.

In [ ]:
from homer_ai.domain import DOMAIN

total = 0
for path in sorted(DOMAIN.rglob("*")):
    if path.is_file():
        total += path.stat().st_size
        print(
            f"{path.relative_to(DOMAIN).as_posix():30} {path.stat().st_size:>8,} bytes"
        )
print(f"{'all of it':30} {total:>8,} bytes")
print()
print((DOMAIN / "savings" / "easy_access.md").read_text(encoding="utf-8")[:330])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Twenty-one files, about 180 KB together. The two big ones are the generated emails and tickets. Everything else was typed by hand.

The head of each product sheet says which product it is, which version, from which day it applies, and that it's invented. Hold on to *version* and *in force from*. They come back in 10_01.

</details>

### ❓ Why are two of the three examples about financial services?

Because it's a field where a wrong answer costs something, so it shows why all the care is needed. A firm there answers for what its customers are told, whoever or whatever wrote the words. In the UK that's how the regulator's recent publications on AI are summarised: firms remain responsible for the outcomes their customers get through AI-supported products and support ([a law firm's briefing](https://www.freshfields.com/en/our-thinking/briefings/2026/10/ai-in-financial-services-fca-good-and-poor-practice-and-the-road-to-agentic-finance)), and rules like the Consumer Duty and the complaint-handling rules set out what firms owe their customers.

I'm a developer, not a lawyer. I mention such rules loosely, as the reason a design looks the way it does, and you should read the rules themselves before you lean on anything here. Nothing in this chapter makes a system *compliant*, and no model in it decides anything about a customer. It looks things up, it flags, and it hands over. People decide.

The third example is boring on purpose. Every company has an IT service desk, a misrouted ticket costs minutes and not a customer, and that makes it a good first project.

### ❓ What do I need on my computer?

- Nothing new for 10_01, 10_03 and 10_04. They run in a few seconds.
- [10_02](10_02_spotting_complaints.ipynb) fine-tunes GPT-2, so it needs the 548 MB file of chapter 8 in `~/.cache/homer-ai/`. If you did chapter 8 it's already there. If not, the notebook downloads it once. The training takes about 40 seconds on my laptop.
- One optional cell in 10_03 fine-tunes GPT-2 too. Without the file it prints one line and moves on.
- 10_01 describes a script that calls Claude. You don't have to run it. The notebook shows one recorded run.

The finished code lives in `src/homer_ai/domain.py`, and the data in `data/domain/`.

### ❓ How good will the results be?

Here's what I measured. Remember that the data is made up.

| Notebook | What we measure | Result |
|---|---|---|
| 10_01 | the right section is among the 3 the search returns (22 questions) | 19 of 22 |
| 10_01 | Claude answers right and cites the right source: no tools, then with our three tools (29 questions) | 0 of 29, then 29 of 29 |
| 10_01 | a word-list guard hands the 10 must-escalate messages to a person | 7 of 10 |
| 10_02 | complaints caught on 80 test emails: keyword list, then fine-tuned GPT-2 | 20 of 39, then 31 of 39 |
| 10_02 | the same on 24 emails I wrote freely | 0 of 12, then 11 of 12 |
| 10_03 | the right article is among 3 suggestions (26 tickets) | 22 of 26 |
| 10_03 | tickets sent to the right team, on wordings never seen (50 tickets) | 17, then 24, then 30 of 50, as the method improves |

None of these is a number to be proud of or ashamed of. Each one comes with a list of what went wrong, and those lists are the useful part.

### ❓ Which notebook do I open for each idea?

| Notebook | Question it answers |
|---|---|
| [`10_01_support_with_citations.ipynb`](10_01_support_with_citations.ipynb) | How do I answer from product documents, get a rate exactly right, and know when not to answer? |
| [`10_02_spotting_complaints.ipynb`](10_02_spotting_complaints.ipynb) | How do I train a small model to flag complaints and vulnerable customers, and how do I score it honestly? |
| [`10_03_service_desk_triage.ipynb`](10_03_service_desk_triage.ipynb) | How do I route tickets and suggest articles without training anything, and what fools the test? |
| [`10_04_pitfalls_and_next_steps.ipynb`](10_04_pitfalls_and_next_steps.ipynb) | What makes a demo like these naive, and what does a real organisation add? |

## 4. Recap

- *Our own LLM* nearly always means rung 2 or 3: a model that answers from our documents, or a small one on our machines.
- Firms climb for exact cited answers, for data that can't leave, for cost at volume, and for a model that stays frozen. Training from scratch is rare.
- The pattern is the one you know: your data, a tool, an MCP server, and a score with and without.
- All data in this chapter is invented, so the numbers show a method and nothing more.

**Next:** [10_01: Support answers with citations](10_01_support_with_citations.ipynb).